In [ ]:
import json
import requests
import pandas as pd

BASE = "http://localhost:16686"
SERVICE = "AdaptivePublisher"   # exact as in /api/services
LIMIT = 200000


OPERATIONS = {"encode_and_store_frame", "process_next_frame"}

# Filter out process_next_frame spans longer than this
LONG_PROCESS_THRESHOLD_MS = 50_000  # 50 seconds


def get_services():
    r = requests.get(f"{BASE}/api/services", timeout=10)
    r.raise_for_status()
    return r.json()["data"]


def fetch_traces(service: str, limit=1500):
    params = {"service": service, "limit": limit}
    r = requests.get(f"{BASE}/api/traces", params=params, timeout=30)
    r.raise_for_status()
    return r.json()["data"]


def p5(s: pd.Series) -> float:
    s = s.dropna()
    return float(s.quantile(0.05)) if len(s) else float("nan")


def p95(s: pd.Series) -> float:
    s = s.dropna()
    return float(s.quantile(0.95)) if len(s) else float("nan")


def round_numeric_df(df: pd.DataFrame, decimals: int = 3) -> pd.DataFrame:
    """Round all numeric columns of a dataframe to a fixed number of decimals."""
    if df is None or df.empty:
        return df
    num_cols = df.select_dtypes(include="number").columns
    if len(num_cols) > 0:
        df[num_cols] = df[num_cols].round(decimals)
    return df


def spans_to_df(traces):
    rows = []
    for t in traces:
        trace_id = t.get("traceID")
        for sp in t.get("spans", []):
            op = sp.get("operationName")
            if op not in OPERATIONS:
                continue

            tags = {x["key"]: x.get("value") for x in sp.get("tags", [])}
            refs = sp.get("references", []) or []

            rows.append({
                "trace_id": trace_id,
                "span_id": sp.get("spanID"),
                "operation_name": op,

                "start_time_raw": sp.get("startTime"),
                "duration_raw": sp.get("duration"),

                "parent_span_id": refs[0]["spanID"] if refs else None,
                "has_parent": len(refs) > 0,
         
                "frame_index": tags.get("frame.index"),
                "image_mode": tags.get("image.mode"),

                "encode_ms": tags.get("encode.ms"),
                "upload_ms": tags.get("upload.ms"),
                "baseline_upload_ms": tags.get("baseline.upload.ms"),
                "payload_bytes": tags.get("payload.bytes"),
            })

    df = pd.DataFrame(rows)

    for c in [
        "frame_index",
        "encode_ms",
        "upload_ms",
        "baseline_upload_ms",
        "payload_bytes",
        "start_time_raw",
        "duration_raw",
    ]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

 
    median_dur = df["duration_raw"].dropna().median()
    if pd.notna(median_dur) and median_dur > 10_000_000:
        df["duration_ms"] = df["duration_raw"] / 1_000_000.0  # ns -> ms
        df["duration_unit_inferred"] = "ns"
    else:
        df["duration_ms"] = df["duration_raw"] / 1_000.0      # us -> ms
        df["duration_unit_inferred"] = "us"

    # Baseline uses baseline.upload.ms; other modes use upload.ms
    df["effective_upload_ms"] = df["upload_ms"].fillna(df["baseline_upload_ms"])
    df["payload_kb"] = df["payload_bytes"] / 1024.0

    return df


def summarize_encode(df_encode):
    """
    Summary per image_mode for encode_and_store_frame spans.
    Includes mean, p5, p95 for encode/upload and payload.
    """
    df = df_encode.dropna(subset=["image_mode"]).copy()

    summary = (
        df.groupby("image_mode")
          .agg(
              frames=("trace_id", "count"),

              encode_ms_mean=("encode_ms", "mean"),
              encode_ms_p5=("encode_ms", p5),
              encode_ms_p95=("encode_ms", p95),

              upload_ms_mean=("effective_upload_ms", "mean"),
              upload_ms_p5=("effective_upload_ms", p5),
              upload_ms_p95=("effective_upload_ms", p95),

              payload_kb_mean=("payload_kb", "mean"),
              payload_kb_p5=("payload_kb", p5),
              payload_kb_p95=("payload_kb", p95),
          )
          .sort_values("frames", ascending=False)
    )
    return summary


def summarize_process_next_frame_overall(df_all):
    """
    One-row overall summary for process_next_frame span duration (ms).
    """
    d = df_all[df_all["operation_name"] == "process_next_frame"].copy()
    if d.empty:
        return pd.DataFrame([{
            "spans": 0,
            "duration_ms_mean": float("nan"),
            "duration_ms_p5": float("nan"),
            "duration_ms_p95": float("nan"),
        }])

    return pd.DataFrame([{
        "spans": int(d["span_id"].count()),
        "duration_ms_mean": float(d["duration_ms"].mean()),
        "duration_ms_p5": p5(d["duration_ms"]),
        "duration_ms_p95": p95(d["duration_ms"]),
    }])


def attach_process_duration_to_encode(df_all):
    """
    Join encode spans to their parent process_next_frame span via references:
      encode.parent_span_id == process.span_id
    Returns encode dataframe with an extra column process_next_frame_ms.
    """
    enc = df_all[df_all["operation_name"] == "encode_and_store_frame"].copy()
    proc = df_all[df_all["operation_name"] == "process_next_frame"][["span_id", "duration_ms"]].copy()
    proc = proc.rename(columns={"span_id": "parent_span_id", "duration_ms": "process_next_frame_ms"})

    out = enc.merge(proc, on="parent_span_id", how="left")
    return out


def summarize_process_next_frame_by_codec(joined_encode):
    """
    If the join worked, summarize process_next_frame duration per image_mode.
    """
    d = joined_encode.dropna(subset=["image_mode"]).copy()
    return (
        d.groupby("image_mode")
         .agg(
             frames=("trace_id", "count"),
             process_next_frame_ms_mean=("process_next_frame_ms", "mean"),
             process_next_frame_ms_p5=("process_next_frame_ms", p5),
             process_next_frame_ms_p95=("process_next_frame_ms", p95),
         )
         .sort_values("frames", ascending=False)
    )


def main():
    services = get_services()
    print("Services in Jaeger:", services)

    if SERVICE not in services:
        raise RuntimeError(f"Service '{SERVICE}' not found in Jaeger")

    traces = fetch_traces(SERVICE, limit=LIMIT)
    print(f"Fetched {len(traces)} traces for service '{SERVICE}'")

    out_json = f"traces_{SERVICE}.json"
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(traces, f)
    print(f"Saved traces to {out_json}")

    # Build dataframe from traces
    df = spans_to_df(traces)
    print(f"Extracted spans (selected ops): {len(df)}")
    if df.empty:
        print("No matching spans found. Check operation names or ensure that code path ran.")
        return

    # Round ALL numeric columns globally to 3 decimals
    df = round_numeric_df(df, 3)

    # Save raw extracted spans
    df.to_csv("spans_all_selected_ops.csv", index=False)
    print("Wrote spans_all_selected_ops.csv")

    # -------------------------------------------------
    # Filter out process_next_frame spans longer than 50s
    # -------------------------------------------------
    df_proc_long = df[
        (df["operation_name"] == "process_next_frame") &
        (df["duration_ms"] > LONG_PROCESS_THRESHOLD_MS)
    ].copy()

    df_proc_long = round_numeric_df(df_proc_long, 3)

    print(f"process_next_frame spans > 50s: {len(df_proc_long)}")
    if not df_proc_long.empty:
        df_proc_long.to_csv("process_next_frame_over_50s.csv", index=False)
        print("Wrote process_next_frame_over_50s.csv")

    # Exclude them from downstream summaries
    df = df[~(
        (df["operation_name"] == "process_next_frame") &
        (df["duration_ms"] > LONG_PROCESS_THRESHOLD_MS)
    )].copy()
    df = round_numeric_df(df, 3)
    print(f"Spans after filtering long process_next_frame: {len(df)}")

    # --- Summary: encode spans per codec ---
    df_encode = df[df["operation_name"] == "encode_and_store_frame"].copy()
    if df_encode.empty:
        print("\nNo encode_and_store_frame spans found.")
    else:
        summary_encode = summarize_encode(df_encode)
        summary_encode = round_numeric_df(summary_encode, 3)

        print("\n=== Summary by image_mode (encode_and_store_frame) ===")
        print(summary_encode)
        summary_encode.to_csv("summary_by_image_mode_encode.csv")
        print("Wrote summary_by_image_mode_encode.csv")

    # --- Summary: process_next_frame overall ---
    summary_proc = summarize_process_next_frame_overall(df)
    summary_proc = round_numeric_df(summary_proc, 3)

    print("\n=== Summary (process_next_frame overall) ===")
    print(summary_proc)
    summary_proc.to_csv("summary_process_next_frame.csv", index=False)
    print("Wrote summary_process_next_frame.csv")


if __name__ == "__main__":
    main()